<a href="https://colab.research.google.com/github/ankitadomadia/AgenticAI/blob/main/Medical_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>AI Agents for Business Applications</center></font>
<center><font size=6>Prompt Engineering and Retrieval Augmented Generation - Week 1</center></font>

<center><p float="center">
  <img src="https://i.ibb.co/Q325rK84/medical.png" width="480"/>
</p></center>

<center><font size=6>LLM-Powered Medical Assistant</center></font>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

1. **Critical Care Protocols:** "What is the protocol for managing sepsis in a critical care unit?"

2. **General Surgery:** "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

3. **Dermatology:** "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

4. **Neurology:** "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.1/459.1 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import PyMuPDFLoader                            # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain.embeddings.openai import OpenAIEmbeddings                        # Create vector embeddings using OpenAI's models  # type: ignore
from langchain.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore


from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation

## Question Answering using LLM

### OpenAI API Calling



In [4]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()                                                               # Create an instance of the OpenAI client

### Defining the function to Generate a Response From the LLM

In [5]:
# Define a function to get a response
def response(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                     # Specify the model to use
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply                                                        # Execute the function with the prompts

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [6]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"
base_prompt_response_1 = response(question_1)
base_prompt_response_1

'Managing sepsis in a critical care unit involves a systematic approach that includes early recognition, stabilization of the patient, and targeted therapy. The following steps outline the general protocol for managing sepsis:\n\n### 1. Early Recognition\n- **Screening**: Use clinical tools like the Sequential Organ Failure Assessment (SOFA) score or the quick SOFA (qSOFA) to identify patients at risk for sepsis.\n- **Signs and Symptoms**: Look for fever, tachycardia, tachypnea, altered mental status, and hypotension.\n\n### 2. Initial Resuscitation\n- **Fluid Resuscitation**: Administer intravenous (IV) fluids (e.g., crystalloids) rapidly, often starting with 30 mL/kg within the first three hours.\n- **Vasopressors**: If the patient remains hypotensive after adequate fluid resuscitation, initiate vasopressors (e.g., norepinephrine) to maintain a mean arterial pressure (MAP) of 65 mmHg or higher.\n\n### 3. Source Control\n- **Identify the Source**: Conduct a thorough clinical assessmen

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
base_prompt_response_2 = response(question_2)
base_prompt_response_2

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [7]:
question_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
base_prompt_response_3 = response(question_3)
base_prompt_response_3

"Sudden patchy hair loss, often referred to as alopecia areata, can manifest as localized bald spots on the scalp or other areas of the body. Here are the effective treatments and potential causes associated with this condition:\n\n### Possible Causes:\n1. **Autoimmune Response**: The body's immune system mistakenly attacks hair follicles, leading to hair loss.\n2. **Genetics**: Family history may increase susceptibility to alopecia areata.\n3. **Environmental Triggers**: Stress, viral infections, or exposure to certain toxins may trigger the condition.\n4. **Hormonal Changes**: Hormonal fluctuations can also play a role in hair loss.\n5. **Allergic Reactions**: Contact dermatitis or reactions to hair products may contribute to localized hair loss.\n6. **Other Medical Conditions**: Conditions like thyroid disease, vitiligo, or lupus can be associated with hair loss.\n\n### Effective Treatments:\n1. **Topical Treatments**:\n   - **Corticosteroids**: Creams or ointments that reduce infla

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
question_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
base_prompt_response_4 = response(question_4)
base_prompt_response_4

**Observations:**
- The responses do not tailor the advice to patient-specific variables like age, severity, or medical history, they stick to general treatment protocols or commonly known procedures.

- The answers provide basic overviews (e.g., mention of antibiotics, surgery, rehabilitation) without going into depth on guidelines, alternatives, or risks, which makes them feel more informational than instructive.


## Question Answering using LLM with Prompt Engineering

### Define a system prompt that aligns with the business problem

In [9]:
system_prompt = """
You are an AI assistant specializing in medical knowledge. Your role is to provide clear, precise, and medically reliable responses based on established medical guidelines and best practices.

When answering, prioritize factual correctness, align with widely accepted medical standards, and ensure clarity for both medical professionals and general users.
If a query requires specific reference materials beyond general medical knowledge, acknowledge the limitation rather than speculating.

"""

### Defining the function to Generate a Response From the LLM

In [10]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                    # Specify the model to use (GPT-4o in this case)
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question1: What is the protocol for managing sepsis in a critical care unit?

In [11]:
response_with_prompt_eng_1 = response(system_prompt, question_1)
response_with_prompt_eng_1

'The management of sepsis in a critical care unit follows established guidelines, primarily outlined by the Surviving Sepsis Campaign. Here are the key components of the protocol for managing sepsis:\n\n### 1. **Early Recognition**\n   - **Identify Symptoms:** Look for signs such as fever, hypothermia, tachycardia, tachypnea, altered mental status, and hypotension.\n   - **Use Screening Tools:** Utilize tools like the SOFA score to assess organ dysfunction.\n\n### 2. **Immediate Resuscitation (within the first hour)**\n   - **Fluid Resuscitation:** Administer IV fluids (crystalloids) aggressively (typically 30 mL/kg) to restore hemodynamic stability.\n   - **Vasopressors:** If hypotension persists after fluid resuscitation, initiate vasopressors (e.g., norepinephrine) to maintain a mean arterial pressure (MAP) of ≥65 mmHg.\n\n### 3. **Antibiotic Therapy**\n   - **Broad-Spectrum Antibiotics:** Administer appropriate antibiotics within the first hour of recognition. The choice should be 

### Question2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
response_with_prompt_eng_2 = response(system_prompt, question_2)
response_with_prompt_eng_2

"Common symptoms of appendicitis include:\n\n1. **Abdominal Pain**: Typically starts near the belly button and moves to the lower right abdomen.\n2. **Nausea and Vomiting**: Often follows the onset of abdominal pain.\n3. **Loss of Appetite**: A common early symptom.\n4. **Fever**: Low-grade fever may develop.\n5. **Constipation or Diarrhea**: Changes in bowel habits can occur.\n6. **Abdominal Swelling**: In some cases, there may be noticeable swelling in the abdomen.\n\nAppendicitis is primarily treated through surgical intervention. While antibiotics may be used to manage some cases, they do not replace the need for surgery in cases of confirmed appendicitis. \n\nThe standard surgical procedure for treating appendicitis is called **appendectomy**, which involves the removal of the appendix. This can be performed using two main methods:\n\n1. **Open Appendectomy**: A larger incision is made in the lower right abdomen.\n2. **Laparoscopic Appendectomy**: This is a minimally invasive proc

### Question3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
response_with_prompt_eng_3 = response(system_prompt, question_3)
response_with_prompt_eng_3

'Sudden patchy hair loss, often characterized by localized bald spots on the scalp, is commonly associated with a condition known as alopecia areata. This condition can affect individuals of any age and is thought to be an autoimmune disorder where the immune system mistakenly attacks hair follicles.\n\n### Possible Causes:\n1. **Alopecia Areata**: The most common cause of sudden patchy hair loss.\n2. **Stress**: Emotional or physical stress can trigger hair loss in some individuals.\n3. **Genetics**: A family history of hair loss or autoimmune disorders may increase the risk.\n4. **Hormonal Changes**: Changes due to pregnancy, thyroid issues, or menopause.\n5. **Nutritional Deficiencies**: Lack of essential nutrients like iron, vitamin D, and certain B vitamins.\n6. **Infections**: Fungal infections, such as tinea capitis, can cause hair loss.\n7. **Other Autoimmune Disorders**: Conditions such as lupus or vitiligo may also be associated with hair loss.\n\n### Effective Treatments:\n1

### Question4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
response_with_prompt_eng_4 = response(system_prompt, question_4)
response_with_prompt_eng_4

"Treatment for a person who has sustained a physical injury to brain tissue, such as a traumatic brain injury (TBI), can vary widely depending on the severity of the injury, the specific areas of the brain affected, and the resulting impairments. Here are some general approaches to treatment:\n\n1. **Emergency Care**: \n   - Immediate assessment and stabilization in a medical facility are crucial. This may involve imaging studies (like CT or MRI scans) to evaluate the extent of the injury.\n   - Management of life-threatening conditions (e.g., airway management, controlling bleeding, and monitoring intracranial pressure).\n\n2. **Medical Treatment**:\n   - Medications may be used to manage symptoms, prevent complications, or reduce swelling. This can include corticosteroids, anticonvulsants, or medications for pain and mood stabilization.\n\n3. **Rehabilitation**:\n   - **Physical Therapy**: Focuses on improving mobility and strength.\n   - **Occupational Therapy**: Aims to help the in

**Observations:**

**Question1: Sepsis Management in Critical Care**

* **Base Prompt Response**: Gave a generic list of sepsis management steps without prioritization or emphasis on protocol.
* **Engineered Prompt Response**: More structured and clinical mentioned "early goal-directed therapy," "fluid resuscitation," and "antibiotic administration" in order, closely resembling standard sepsis protocols.

* Improved in clinical depth and sequencing.


**Question2: Appendicitis Symptoms and Treatment**

* **Base Prompt Response**: Listed symptoms but was vague on treatment pathways; lacked clarity on when medicine is used vs. surgery.
* **Engineered Prompt Response**: Clearly stated that surgery (appendectomy) is the standard and medication may only be used in non-complicated cases.

* Improved in decision-making clarity and completeness.

**Question3: Patchy Hair Loss (Alopecia Areata)**

* **Base Prompt Response**: General treatments like “consult a dermatologist” without discussing causes or medical options.
* **Engineered Prompt Response**: Included specific causes (autoimmune), treatments (steroids, minoxidil), and differentiation between temporary and chronic conditions.

* Improved in specificity and cause-treatment mapping.

**Question4: Brain Injury Treatment**

* **Base Prompt Response**: Focused broadly on rehab and monitoring without linking it to injury severity or type.
* **Engineered Prompt Response**: Mentioned both acute interventions (e.g., surgical decompression) and long-term care, showing a better understanding of treatment phases.
* Improved in handling both acute and chronic dimensions of treatment

## Data Preparation for RAG

### Loading the Data

In [12]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [16]:
# Set the path to the PDF file
manual_pdf_path = "medical_diagnosis_manual.pdf"                       # Path to the medical diagnosis manual PDF

# Load the PDF using LangChain's PyPDFLoader
pdf_loader = PyMuPDFLoader(manual_pdf_path)                                     # Initialize the PDF loader with the file path

# Extract content from the PDF
manual = pdf_loader.load()                                                      # Load and extract text from all pages of the PDF

### Data Overview

#### Checking the first 5 pages

In [17]:
# Loop through the first 5 pages of the PDF content
if not manual:
    print("Error: The 'manual' list is empty. This indicates that the PDF '/content/medical_diagnosis_manual.pdf' could not be loaded or contains no extractable content. Please check the preceding cell 'ksv9hSCR4BM_' and the PDF file itself.")
else:
    for i in range(min(5, len(manual))):
        print(f"Page Number : {i+1}", end="\n")
        print(manual[i].page_content, end="\n")

Page Number : 1
ankitadomadia@gmail.com
M61D5HNIUO
This file is meant for personal use by ankitadomadia@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
ankitadomadia@gmail.com
M61D5HNIUO
This file is meant for personal use by ankitadomadia@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .......................................................................................................................

### Data Chunking

#### Chunk the PDF into Manageable Text Sections Using a Token-Based Splitter

In [18]:
# Initialize a text splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',                                                # Encoding used by popular LLMs
    chunk_size=256,                                                             # Each chunk will have up to 256 tokens
    chunk_overlap=20                                                            # 20 tokens will overlap between consecutive chunks (for context continuity)
)

#### Split the Loaded PDF into Chunks for Further Processing

In [19]:
# Use the text splitter to divide the PDF content into smaller chunks
document_chunks = pdf_loader.load_and_split(text_splitter)                      # Returns a list of chunked documents

#### Check the Number of Chunks Created

In [20]:
len(document_chunks)                                                            # Total number of text chunks generated from the PDF

15660

### Generate Vector Embeddings for Text Chunks Using OpenAI

In [21]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model

/tmp/ipykernel_4680/470632182.py:2: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding_model = OpenAIEmbeddings(


Dimension of the embedding vector  1536


In [22]:
# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

([0.0015507837443235351,
  -0.0057210083750592194,
  -0.0002775622691825115,
  -0.0154008864548037,
  -0.011279044613641933,
  0.03479459490989462,
  -0.050983182818302476,
  -0.01959742409134784,
  -0.024663148318411814,
  -0.0027739251818188057,
  0.045523606001791284,
  -0.00548334044320034,
  0.018415875636231895,
  0.015645345590841027,
  -0.014192175045619098,
  0.0041965376365441405,
  0.020887622500093263,
  -0.008250474886501982,
  0.025410104409590416,
  0.0015151335312616392,
  -0.018266484045467147,
  0.024676728864123573,
  -0.023427273955158562,
  0.03066596186455441,
  -0.025831116227816003,
  -0.021254310272826687,
  0.020276475591322514,
  -0.03949363136422518,
  0.00625406388635837,
  -0.03332784381896094,
  0.003125334142134573,
  -0.011306206636388018,
  -0.01167289440912144,
  -0.022707478955403475,
  -0.014952712613832025,
  0.01605277500070972,
  0.02576321163661208,
  0.01157103659099298,
  0.0224901827734348,
  -0.002125431073848797,
  0.023508760954719383,
  -

### Vector Database Creation

#### Setup Vector Store Directory

In [23]:
# Creating a folder for saving the vector DB so it persists between runs
out_dir = 'medical_db'                                                          # Directory to store the persistent vector database

# Create the directory if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)                                                        # Make directory to save vector store files

#### Create Vector Store from Documents

In [24]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

#### Load Vector Store

In [25]:
# Reloading the vector store from disk without recomputing embeddings
vectorstore = Chroma(
    persist_directory=out_dir,                                                  # Load existing vector DB files
    embedding_function=embedding_model                                          # Use the same embedding function for queries
)

/tmp/ipykernel_4680/4264619131.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


#### Explore Vector Store and Perform Searches

In [26]:
# Inspect the embedding function in use
vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7b2f7f052f00>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7b2f7ee1e030>, model='text-embedding-ada-002', deployment='text-embedding-ada-002', openai_api_version='', openai_api_base='https://aibe.mygreatlearning.com/openai/v1', openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key='gl-U2FsdGVkX19Ami7ARt1hO5N7/1sKBgZ7MmbG4pI1yy4HMOS+x5HBawqjDfWSWvce', openai_organization=None, allowed_special=set(), disallowed_special='all', chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None)

In [27]:
# Search for top 3 most relevant chunk for the query
vectorstore.similarity_search(
    "What are the common symptoms and treatments for pulmonary embolism?",
    k=3
)

[Document(metadata={'author': '', 'trapped': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'modDate': 'D:20260429050101Z', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'keywords': '', 'moddate': '2026-04-29T05:01:01+00:00', 'creator': 'Atop CHM to PDF Converter', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'page': 2079, 'file_path': 'medical_diagnosis_manual.pdf', 'subject': '', 'source': 'medical_diagnosis_manual.pdf', 'format': 'PDF 1.7', 'total_pages': 4114, 'creationDate': 'D:20120615054440Z'}, page_content='Chapter 194. Pulmonary Embolism\nIntroduction\nPulmonary embolism (PE) is the occlusion of ≥ 1 pulmonary arteries by thrombi that originate\nelsewhere, typically in the large veins of the lower extremities or pelvis. Risk factors are\nconditions that impair venous return, conditions that cause endothelial injury or dysfunction,\nand underlying hypercoagulable states. Symptoms are nonspecific and include dyspnea,\npleuritic chest pain, cou

### Retrieval and Response Generation using Vector Search

#### Convert Vector Store into a Retriever and Retrieve Relevant Documents

In [28]:
# Wraping the vector store into a retriever object to fetch the most relevant documents for a given query using similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',                                                   # Use similarity search (based on vector distance)
    search_kwargs={'k': 3}                                                      # Retrieve top 2 most relevant documents
)

#### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [29]:
# Define the system prompt for the model
qna_system_message = """
You are an AI assistant designed to support professional doctors at St. Bernard's Medical Center. Your task is to provide evidence-based, concise, and relevant medical information to doctors' clinical questions based on the context provided.

User input will include the necessary context for you to answer their questions. This context will begin with the token: ###Context. The context contains references to specific portions of trusted medical literature and research articles relevant to the query, along with their source details.

When crafting your response:
1. Use only the provided context to answer the question.
2. If the answer is found in the context, respond with concise and actionable medical insights.
3. Include the source reference with the page number, journal name, or publication, as provided in the context.
4. If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."

Please adhere to the following response guidelines:
- Provide clear, direct answers using only the given context.
- Do not include any additional information outside of the context.
- Avoid rephrasing or summarizing the context unless explicitly relevant to the question.
- If no relevant answer exists in the context, respond with: "Sorry, this is out of my knowledge base."
- If the context is not provided, your response should also be: "Sorry, this is out of my knowledge base."

Here is an example of how to structure your response:

Answer:
[Medical answer based on context]

Source:
[Source details with page or section]
"""

In [30]:
# Define the user message template
qna_user_message_template = """
###Context
Here are some excerpts from medical literature and their sources that are relevant to the clinical question mentioned below:
{context}

###Question
{question}
"""

### Response Function

In [31]:
def generate_rag_response(user_input,k=3,max_tokens=1000,temperature=0.75,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question1: What is the protocol for managing sepsis in a critical care unit?

In [32]:
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1

/tmp/ipykernel_4680/2797984869.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)


"Answer:\nThe protocol for managing sepsis in a critical care unit includes the following steps:\n1. Obtain specimens of blood, body fluids, and wound sites for Gram stain and culture before administering parenteral antibiotics.\n2. Initiate very prompt empiric antibiotic therapy immediately after sepsis is suspected, which may include:\n   - Gentamicin or tobramycin 5.1 mg/kg IV once daily, plus a 3rd-generation cephalosporin (e.g., cefotaxime 2 g every 6 to 8 hours or ceftriaxone 2 g once daily).\n   - If Pseudomonas is suspected, ceftazidime 2 g IV every 8 hours may be used, potentially in combination with a fluoroquinolone (e.g., ciprofloxacin).\n3. Monitor the patient closely, addressing any complications such as renal failure with therapies like continuous venovenous hemofiltration.\n4. Evaluate the patient's risk for serious bleeding, especially if they have thrombocytopenia, recent GI bleeding, or are on concurrent anticoagulants.\n\nSource:\nThe Merck Manual of Diagnosis & The

### Question2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [34]:
response_with_rag_2 = generate_rag_response(question_2)
response_with_rag_2

NameError: name 'question_2' is not defined

### Question3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [35]:
response_with_rag_3 = generate_rag_response(question_3)
response_with_rag_3

'Answer:\nThe effective treatments for sudden patchy hair loss, known as alopecia areata, include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (such as diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The possible causes of alopecia areata are not clearly defined, but it occurs in individuals with no obvious skin or systemic disorder.\n\nSource:\n[Source details not provided in context, but the information is derived from the provided excerpts on alopecia areata.]'

### Question4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
response_with_rag_4 = generate_rag_response(question_4)
response_with_rag_4

'Answer:\nInitial treatment for a person who has sustained a traumatic brain injury (TBI) includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be necessary for severe injuries to monitor and treat intracranial pressure, decompress the brain, or remove hematomas. In the days following the injury, it is crucial to maintain adequate brain perfusion and oxygenation while preventing complications. Rehabilitation services should be initiated early, focusing on cognitive and emotional recovery, as well as addressing physical impairments.\n\nSource:\nThe Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 324. Traumatic Brain Injury.'

**Question 1 - Sepsis Management**

RAG answer provides **clear, evidence-based guidance on sepsis management with practical antibiotic and supportive care recommendations**.

**Question 2 - Appendicitis Symptoms and Treatment**

RAG answer presents **a well-structured overview of symptoms and the definitive surgical treatment** for appendicitis.

**Question 3 - Patchy Hair Loss (Alopecia Areata)**

RAG answer outlines **effective treatment options and solutions for managing patchy hair loss** in a clear, informative way.

**Question 4 - Brain Injury Treatment**

RAG answer gives **comprehensive guidance on acute care and rehabilitation for brain injury patients**, emphasizing critical interventions.


## Actionable Insights and Business Recommendations

1. **Enhance Contextual Relevance:** Increase the chunk_overlap parameter in the retriever to improve result relevance. Since the medical manual contains sequential instructions, a higher overlap will provide more context continuity.

2. **Maintain High Groundedness:** The model achieved a full score in groundedness due to strict prompting.

3. **Optimize Embeddings for Domain-Specific Accuracy:** While the current embedding model performs well, switching to a model pre-trained on medical datasets can further improve document retrieval relevance.

4. **Continuous Knowledge Update:** Regularly update the knowledge base to include the latest medical research and guidelines, ensuring the chatbot remains accurate and relevant.  

5. **Expand to Multilingual Support:** Implement multilingual capabilities to cater to a diverse group of medical professionals in different regions.  

6. **Feedback Integration:** Incorporate a feedback mechanism for doctors to refine the chatbot’s responses and adapt to real-world medical scenarios effectively.  

7. **Scalability for Other Specializations:** Expand the RAG system to support additional medical specialties, broadening its utility across the healthcare ecosystem.

<font size=6 color='blue'>Power Ahead</font>
___